# Figure regeneration for `hardwall_ctqw_pra`

This notebook regenerates the manuscript figures as **PNG** files at **dpi=200** using the **SciencePlots `science`** style and **LaTeX** text rendering.

Outputs: `fig_opt_state.png`, `fig_vmax_scaling.png`, `fig_coherent_incoherent.png`, and (optionally) `fig_opt_bias.png`, `fig_lambda_u.png`.

If your `.tex` includes `.pdf` extensions, an optional cell can patch it to `.png` (with a `.bak` backup).

In [ ]:

# --- 0) Environment & style ---
import os
from pathlib import Path

# Threading: tune for your machine if desired (i9-14900 has many cores).
# Set BEFORE importing numpy/scipy for best effect.
os.environ.setdefault("OMP_NUM_THREADS", "16")
os.environ.setdefault("OPENBLAS_NUM_THREADS", os.environ["OMP_NUM_THREADS"])
os.environ.setdefault("MKL_NUM_THREADS", os.environ["OMP_NUM_THREADS"])
os.environ.setdefault("NUMEXPR_NUM_THREADS", os.environ["OMP_NUM_THREADS"])

# Optional: install SciencePlots if missing (recommended)
try:
    import scienceplots  # noqa: F401
except Exception:
    print("SciencePlots not found. Installing...")
    # If you're offline, this will fail; then install manually: pip install SciencePlots
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "SciencePlots"])

import numpy as np
import scipy.linalg as la
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# SciencePlots + LaTeX
import scienceplots  # noqa: F401
plt.style.use(["science"])

plt.rcParams.update({
    "text.usetex": True,
    # Font sizes: larger than default to be readable in a two-column PRA PDF.
    "font.size": 16,
    "axes.labelsize": 16,
    "axes.titlesize": 14,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 14,
    "lines.linewidth": 2.0,
    "figure.dpi": 200,  # interactive display only
})

print("Threads:", os.environ.get("OMP_NUM_THREADS"))


In [ ]:
# --- 1) Output directory ---
from pathlib import Path

# Figure files are saved beside the notebook when run from the repository.
OUTDIR = Path(".")
OUTDIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTDIR.resolve())


In [ ]:

# --- 2) Analytic construction of A^{(M)} for mean drift (Eq. (Adef)) ---
# For J=1:
#   A_{mn} = (2/pi) ∫_0^π sin[(m+1)k] sin[(n+1)k] * (2 sin k) dk
# We implement a closed-form expression to avoid expensive quadrature.

def _Jint(p):
    # Jint(p) = ∫_0^π sin(p k) dk = (1 - cos(pπ))/p for integer p, with Jint(0)=0.
    # For integer p: Jint(p)=0 if p even; Jint(p)=2/p if p odd. Works for negative p too.
    p = np.asarray(p, dtype=np.int64)
    out = np.zeros_like(p, dtype=np.float64)
    mask = (p != 0) & (np.abs(p) % 2 == 1)
    out[mask] = 2.0 / p[mask]
    return out

def A_mean_drift(M, J=1.0):
    # Build the MxM symmetric matrix A^{(M)} for mean drift.
    a = (np.arange(M) + 1).astype(np.int64)  # a = m+1
    b = (np.arange(M) + 1).astype(np.int64)  # b = n+1

    Aplus = a[:, None] + b[None, :]
    Adiff = a[:, None] - b[None, :]

    term = (_Jint(Adiff + 1) + _Jint(-Adiff + 1) + _Jint(Aplus - 1) - _Jint(Aplus + 1))
    A = (J / np.pi) * term
    A = 0.5 * (A + A.T)
    return A

# Optional sanity check (small M) against a brute-force quadrature.
def A_mean_drift_quadrature(M, J=1.0, Nk=6001):
    k = np.linspace(0.0, np.pi, Nk)
    dk = k[1] - k[0]
    w = (2.0 / np.pi) * (2.0 * J * np.sin(k)) * dk
    n = np.arange(M) + 1
    S = np.sin(np.outer(k, n))  # (Nk, M)
    A = (S * w[:, None]).T @ S
    return 0.5 * (A + A.T)

M_test = 8
A_an = A_mean_drift(M_test, J=1.0)
A_q  = A_mean_drift_quadrature(M_test, J=1.0)
err = np.max(np.abs(A_an - A_q))
print("max|A_analytic - A_quadrature| for M=8:", err)


In [ ]:
# --- 3) Figure: Optimal state for mean drift (fig_opt_state.png) ---
J = 1.0
M = 200

A = A_mean_drift(M, J=J)

# principal eigenpair (largest eigenvalue)
evals, evecs = la.eigh(A)
vmax = float(evals[-1])
psi = evecs[:, -1].copy()

# Fix a global phase/sign for reproducibility
if psi[0] < 0:
    psi *= -1

prob = psi**2

# Spectral weight near k0=pi/2 in scaled variable x = M(k - pi/2)
xmax = 12.0
Nx = 2401
x = np.linspace(-xmax, xmax, Nx)
k = (np.pi/2) + x / M

# NOTE: sine basis uses n=1..M if your hard-wall sites are labeled n=1,2,...
n = np.arange(M) + 1
S = np.sin(np.outer(k, n))  # (Nx, M)
c = np.sqrt(2.0/np.pi) * (S @ psi)
spec = np.abs(c)**2  # unnormalized (peak grows ~ O(M) in x-variable)
spec /= spec.max()

# --- measure FWHM in x and annotate Δk = Δx/M on the plot ---
peak = float(np.max(spec))
level = 0.5 * peak
mask = spec >= level

xL = xR = None
dx = dk = None
if np.any(mask):
    iL = int(np.argmax(mask))
    iR = int(len(mask) - 1 - np.argmax(mask[::-1]))

    # linear interpolation for left crossing
    if iL > 0:
        x0, x1 = x[iL - 1], x[iL]
        y0, y1 = spec[iL - 1], spec[iL]
        xL = x0 + (level - y0) * (x1 - x0) / (y1 - y0 + 1e-300)
    else:
        xL = float(x[iL])

    # linear interpolation for right crossing
    if iR < len(x) - 1:
        x0, x1 = x[iR], x[iR + 1]
        y0, y1 = spec[iR], spec[iR + 1]
        xR = x0 + (level - y0) * (x1 - x0) / (y1 - y0 + 1e-300)
    else:
        xR = float(x[iR])

    dx = float(xR - xL)       # FWHM in x
    dk = dx / float(M)        # corresponding width in k

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 3.2), constrained_layout=True)
import matplotlib.ticker as mticker

ax1.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
ax1.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
ax1.yaxis.get_major_formatter().set_powerlimits((0, 0))

ax1.plot(np.arange(1, M + 1), prob, linewidth=1)
ax1.set_xlabel(r"$n$", fontsize=20)
ax1.set_ylabel(r"$|\psi_n|^2$", fontsize=20)

ax2.plot(x, spec, linewidth=2)
ax2.set_xlabel(r"$x$", fontsize=20)
ax2.set_ylabel(r"$|c_\psi(k)|^2$", fontsize=20)
ax2.set_xlim(-xmax, xmax)

# Add bandwidth annotation on panel (b)
if (xL is not None) and (xR is not None) and (dx is not None) and (dk is not None):
    ax2.axvline(xL, ls="--", lw=1.2, color="black")
    ax2.axvline(xR, ls="--", lw=1.2, color="black")

    # place arrow low enough to avoid the peak; scale with peak height
    y_ann = 0.5 * peak
    ax2.annotate(
        "", xy=(xR, y_ann), xytext=(xL, y_ann),
        arrowprops=dict(arrowstyle="<->", lw=1.4)
    )
    ax2.text(
        6, y_ann,
        rf"$\Delta k\approx \frac{{{dx:.2f}}}{{M}}$",
        ha="center", va="bottom", fontsize=16
    )

# panel labels: (a), (b) at top-left of each subfigure
for ax, lab in [(ax1, "(a)"), (ax2, "(b)")]:
    ax.text(
        -0.25, 1.05, lab,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=16,
        fontweight="bold",
    )

for ax in (ax1, ax2):
    for sp in ax.spines.values():
        sp.set_linewidth(1.2)
    ax.tick_params(width=1.2, labelsize=18)

outpath = OUTDIR / "fig_opt_state.png"
fig.savefig(outpath, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", outpath.resolve())
print("v_max(M=200) =", vmax)
if dx is not None:
    print(f"FWHM: Δx ≈ {dx:.6g},  Δk ≈ {dk:.6g} (should scale ~ 1/M)")


In [ ]:
# --- 4) Figure: Optimized mean drift scaling (clean single main + small inset) ---
M_list = [40, 60, 80, 100, 120, 150, 180, 200, 240, 280, 320, 500, 1000,10000]
vmax_list = []

for Mm in tqdm(M_list, desc="Diagonalizing A^{(M)}"):
    A_m = A_mean_drift(Mm, J=J)
    top = la.eigh(A_m, eigvals_only=True, subset_by_index=[Mm-1, Mm-1])[0]
    vmax_list.append(float(top))

M_arr = np.array(M_list, dtype=np.float64)
vmax_arr = np.array(vmax_list, dtype=np.float64)

defect = 2.0 * J - vmax_arr
invM2 = 1.0 / (M_arr**2)

C1 = (np.pi**2) / 4.0  # asymptote for M^2(2J-vmax)/J

import matplotlib.ticker as mticker
from matplotlib.ticker import AutoMinorLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

scaled = (M_arr**2) * defect / J           # -> pi^2/4
residual = C1 - scaled                     # -> 0
x_res = 1.0 / M_arr

# Optional: show also defect vs 1/M^2 as a second inset (very small)
SHOW_DEFECT_INSET = False

# Fit residual ~ a*(1/M) + b using large-M subset (visual guide only)
M_fit_min = max(120, int(0.4 * M_arr.max()))
fit_mask = M_arr >= M_fit_min
p1, p0 = np.polyfit(x_res[fit_mask], residual[fit_mask], 1)
xres_line = np.linspace(0.0, x_res.max() * 1.02, 200)
res_line = p1 * xres_line + p0

# Theory guide for defect vs 1/M^2 (only used if SHOW_DEFECT_INSET=True)
xline = np.linspace(0.0, invM2.max() * 1.05, 300)
y_theory = (C1 * J) * xline


In [ ]:

fig, ax = plt.subplots(1, 1, figsize=(5.5, 4))
fig.subplots_adjust(left=0.12, right=0.98, bottom=0.22, top=0.94)

# ---- main plot: saturation to pi^2/4 ----
ax.plot(M_arr, scaled, marker="o", linestyle="-", linewidth=2.6, markersize=5.5,
        label=r"numerics")
ax.axhline(C1, linestyle="--", linewidth=2.2, color="black",
           label=r"$\pi^2/4$")
ax.set_xscale('log') 
ax.set_xlabel(r"$M$", fontsize=20)
ax.set_ylabel(r"$M^2(2-v_{\rm max}(M))$", fontsize=20)

ax.xaxis.set_minor_locator(AutoMinorLocator(2))
ax.yaxis.set_minor_locator(AutoMinorLocator(2))

for sp in ax.spines.values():
    sp.set_linewidth(1.2)
ax.tick_params(width=1.2, labelsize=16, direction="in", top=True, right=True)

# Keep legend compact and out of the data
ax.legend(frameon=False, fontsize=16, loc="lower left", bbox_to_anchor=(0.1, 0.02))

# Set y-limits to give the asymptote some headroom but not waste space
ymin = min(np.min(scaled), C1) - 0.015
ymax = max(np.max(scaled), C1) + 0.01
ax.set_ylim(ymin, ymax)

# ---- inset 1: residual vs 1/M (focus on large-M regime) ----
axins1 = inset_axes(
    ax, width="38%", height="38%", loc="center right",
    bbox_transform=ax.transAxes, borderpad=2
)

# choose a large-M window for the inset (controls both what you see and what you fit)
M_inset_min = 200.0   # adjust: 200/300/500 depending on how strict you want "large M"
mask_inset = M_arr >= M_inset_min

x_in = x_res[mask_inset]        # 1/M on large-M subset
y_in = residual[mask_inset]     # residual on large-M subset

# plot only the large-M subset in the inset (this removes the "offset" impression)
axins1.plot(x_in, y_in, marker="o", linestyle="-", linewidth=1.4, markersize=3.2)

# fit guide: force through origin, y ≈ a*(1/M)
# (physically, residual -> 0 as 1/M -> 0, so this is the right visual guide)
a_fit = float(np.dot(x_in, y_in) / (np.dot(x_in, x_in) + 1e-300))
xline_in = np.linspace(0.0, float(x_in.max()) * 1.05, 200)
axins1.plot(xline_in, a_fit * xline_in, linestyle="--", linewidth=1.2)

# cosmetics
axins1.xaxis.set_minor_locator(AutoMinorLocator(2))
axins1.yaxis.set_minor_locator(AutoMinorLocator(2))
for sp in axins1.spines.values():
    sp.set_linewidth(1.0)
axins1.tick_params(width=1.0, labelsize=12, direction="in", top=True, right=True)

axins1.text(
    -0.11, 1.2,
    r"$\pi^2/4 - M^2(2-v_{\max})$ vs $1/M$",
    transform=axins1.transAxes, ha="left", va="top", fontsize=12
)

# set x/y limits to the large-M window so the convergence is visually resolved
axins1.set_xlim(0.0, float(x_in.max()) * 1.05)
ymax_in = float(np.max(y_in))
axins1.set_ylim(0.0, ymax_in * 1.15)

# axins1.ticklabel_format(axis="both", style="sci", scilimits=(0, 0))
# axins1.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
# axins1.yaxis.get_major_formatter().set_powerlimits((0, 0))

outpath = OUTDIR / "fig_vmax_scaling.png"
fig.savefig(outpath, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", outpath.resolve())
print(f"Asymptote pi^2/4 = {C1:.10f}")
print(f"Residual fit (large M): residual ≈ {p1:.3g}*(1/M) + {p0:.3g}, using M >= {M_fit_min}")


## Figure: coherent optimum versus site-diagonal reference

This cell adds the coherence-reference figure used in the revised manuscript. It reuses the same analytic matrix, plotting style, and output directory as the other manuscript figures.

In [ ]:
# --- 4b) Figure: Coherent optimum versus site-diagonal reference (fig_coherent_incoherent.png) ---
# This figure uses the same A^{(M)} matrix as Fig. 4.  The site-diagonal
# incoherent reference is max_n A_{nn}=16J/(3*pi), attained at n=0.
from matplotlib.ticker import AutoMinorLocator, LogLocator, NullFormatter

M_coh = np.arange(1, 69, dtype=int)

vmax_coh = []
for Mm in tqdm(M_coh, desc="Diagonalizing A^{(M)} for coherence reference"):
    A_m = A_mean_drift(int(Mm), J=J)
    top = la.eigh(A_m, eigvals_only=True, subset_by_index=[int(Mm)-1, int(Mm)-1])[0]
    vmax_coh.append(float(top))
vmax_coh = np.array(vmax_coh, dtype=np.float64)

v_inc = 16.0 * J / (3.0 * np.pi)
front = 2.0 * J

# sanity checks: exact small-M agreement, monotonicity, and front bound
assert abs(vmax_coh[0] - v_inc) < 1.0e-12
assert np.all(np.diff(vmax_coh) >= -1.0e-10)
assert np.all(vmax_coh <= front + 1.0e-10)

fig, ax = plt.subplots(1, 1, figsize=(5.5, 4))
fig.subplots_adjust(left=0.12, right=0.98, bottom=0.22, top=0.94)

ax.plot(M_coh, vmax_coh / J, marker="o", linestyle="-", linewidth=2.6, markersize=3.8,
        label=r"$v_{\max}(M)/J$")
ax.axhline(v_inc / J, linestyle="--", linewidth=2.2, color="black",
           label=r"$16/(3\pi)$")
ax.axhline(front / J, linestyle=":", linewidth=2.2, color="black",
           label=r"$2$")

ax.set_xscale("log")
ax.set_xlabel(r"$M$", fontsize=20)
ax.set_ylabel(r"$v/J$", fontsize=20)
ax.set_ylim(v_inc / J - 0.035, front / J + 0.03)

ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1))
ax.xaxis.set_minor_formatter(NullFormatter())
ax.yaxis.set_minor_locator(AutoMinorLocator(2))

for sp in ax.spines.values():
    sp.set_linewidth(1.2)
ax.tick_params(width=1.2, labelsize=16, direction="in", top=True, right=True)
ax.legend(frameon=False, fontsize=16, loc="center right", bbox_to_anchor=(0.98, 0.43))

outpath = OUTDIR / "fig_coherent_incoherent.png"
fig.savefig(outpath, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", outpath.resolve())
print("number of M values:", len(M_coh))
print("v_inc/J:", v_inc / J)
print("vmax(M=1)/J:", vmax_coh[0] / J)
print(f"vmax(M={int(M_coh[-1])})/J:", vmax_coh[-1] / J)


## Optional: extra figures

Set `DO_EXTRA=True` to also generate `fig_opt_bias.png` and `fig_lambda_u.png` (useful if you re-enable those figures in the manuscript).

In [ ]:
DO_EXTRA = True


In [ ]:

# --- 5) Extra Figure: Effect of biasing (fig_opt_bias.png) ---
def Au_defect_bias(M, u, J=1.0, xmax=12.0, Nx=4001):
    x = np.linspace(-xmax, xmax, Nx)
    k = (np.pi/2) + x / M
    dk = (x[1] - x[0]) / M
    weight = np.exp(-u * (M**2) * (2.0 - 2.0*np.sin(k))) * dk  # includes dk
    n = np.arange(M) + 1
    S = np.sin(np.outer(k, n))  # (Nx, M)
    Wsqrt = np.sqrt(weight)
    SW = S * Wsqrt[:, None]
    Au = (2.0/np.pi) * (SW.T @ SW)
    Au = 0.5 * (Au + Au.T)
    return Au

if DO_EXTRA:
    u = 1.0
    Au = Au_defect_bias(M, u=u, J=J, xmax=12.0, Nx=4001)
    evals_u, evecs_u = la.eigh(Au)
    psi_u = evecs_u[:, -1].copy()
    if psi_u[0] < 0:
        psi_u *= -1

    prob_mean = psi**2
    prob_bias = psi_u**2

    xmax_plot = 12.0
    Nx_plot = 2401
    x_plot = np.linspace(-xmax_plot, xmax_plot, Nx_plot)
    k_plot = (np.pi/2) + x_plot / M
    n = np.arange(M) + 1
    S_plot = np.sin(np.outer(k_plot, n))
    c_mean = np.sqrt(2.0/np.pi) * (S_plot @ psi)
    c_bias = np.sqrt(2.0/np.pi) * (S_plot @ psi_u)
    spec_mean = np.abs(c_mean)**2
    spec_bias = np.abs(c_bias)**2
    spec_mean /= spec_mean.max()
    spec_bias /= spec_bias.max()


In [ ]:

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 3.2), constrained_layout=True)

ax1.plot(np.arange(M), prob_mean, label=r"mean-drift optimum", linewidth=1)
ax1.plot(np.arange(M), prob_bias, label=r"defect-bias optimum ($u=1$)", linewidth=1)
ax1.set_xlabel(r"$n$",fontsize=20)
ax1.set_ylabel(r"$|\psi_n|^2$",fontsize=20)
# ax1.legend(loc="upper right", frameon=True)
ax1.ticklabel_format(axis="both", style="sci", scilimits=(0, 0))
ax1.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
ax1.yaxis.get_major_formatter().set_powerlimits((0, 0))

ax2.plot(x_plot, spec_mean, label=r"mean-drift optimum")
ax2.plot(x_plot, spec_bias, label=r"defect-bias optimum ($u=1$)")
ax2.set_xlabel(r"$\xi$",fontsize=20)
ax2.set_ylabel(r"$|c_{\psi(k)}|^2$",fontsize=20)
ax2.set_xlim(-xmax_plot, xmax_plot)
# ax2.legend(loc="upper right", frameon=True)

# Add bandwidth annotation on panel (b)
if (xL is not None) and (xR is not None) and (dx is not None) and (dk is not None):
    ax2.axvline(xL, ls="--", lw=1.2, color="black")
    ax2.axvline(xR, ls="--", lw=1.2, color="black")

    # place arrow low enough to avoid the peak; scale with peak height
    y_ann = 0.5 * peak
    ax2.annotate(
        "", xy=(xR, y_ann), xytext=(xL, y_ann),
        arrowprops=dict(arrowstyle="<->", lw=1.4)
    )
    ax2.text(
        6, y_ann,
        rf"$\Delta k\approx \frac{{{dx:.2f}}}{{M}}$",
        ha="center", va="bottom", fontsize=16
    )

# --- 关键修改：提取图例并合并 ---
# 获取 ax1 的 handles 和 labels (因为两个图 label 一样，取一个即可)
handles, labels = ax1.get_legend_handles_labels()

# 在 Figure 级别添加图例
# loc="upper center" 配合 ncol=2 可以让图例横向排列
fig.legend(
    handles=handles, 
    labels=labels, 
    loc="upper center", 
    ncol=2, 
    frameon=False,
    fontsize=16,
    bbox_to_anchor=(0.55, 1.15)  # 这里的 1.1 表示在图表上方的距离，根据需要微调
)

# panel labels: (a), (b) at top-left of each subfigure
ax1.text(
    -0.18, 1, "(a)",
    transform=ax1.transAxes,
    ha="left", va="top",
    fontsize=16,
    fontweight="bold")

ax2.text(
    -0.23, 1, "(b)",
    transform=ax2.transAxes,
    ha="left", va="top",
    fontsize=16,
    fontweight="bold")

for ax in (ax1, ax2):
    for sp in ax.spines.values():
        sp.set_linewidth(1.2)
    ax.tick_params(width=1.2, labelsize=20)

outpath = OUTDIR / "fig_opt_bias.png"
fig.savefig(outpath, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", outpath.resolve())


In [ ]:

# --- 6) Extra Figure: Universal scaling function lambda(u) (fig_lambda_u.png) ---
def lambda_continuum(u, Nq=320):
    p, w = np.polynomial.legendre.leggauss(Nq)  # nodes/weights on [-1,1]
    pref = (1.0/(2.0*np.pi)) * np.sqrt(np.pi/u)  # matches Ku(p,q) in Eq.(31)
    diff = p[:, None] - p[None, :]
    K = pref * np.exp(-(diff*diff) / (4.0*u))

    # Correct symmetric Nyström matrix: S_ij ≈ sqrt(w_i) K(p_i,p_j) sqrt(w_j)
    sw = np.sqrt(w)
    A = (sw[:, None] * K) * sw[None, :]

    val = la.eigh(A, eigvals_only=True, subset_by_index=[Nq-1, Nq-1])[0]
    return float(val)


u_grid = np.linspace(0.02, 2.0, 50)
lam = np.array([lambda_continuum(float(u), Nq=320) for u in tqdm(u_grid, desc="Continuum lambda(u)")])

# finite-M markers (moderate workload)
M_mark = [80, 120, 200]
u_mark = np.array([0.05, 0.10, 0.20, 0.40, 0.80, 1.20, 1.60])
G = {Mm: [] for Mm in M_mark}
for Mm in M_mark:
    for u in tqdm(u_mark, desc=f"Discrete G_M(u) for M={Mm}"):
        Au_m = Au_defect_bias(Mm, u=float(u), J=J, xmax=12.0, Nx=3001)
        top = la.eigh(Au_m, eigvals_only=True, subset_by_index=[Mm-1, Mm-1])[0]
        G[Mm].append(float(top))
    G[Mm] = np.array(G[Mm])

C1 = (np.pi**2)/4.0



In [ ]:

fig, ax = plt.subplots(1, 1, figsize=(4.6, 3.2), constrained_layout=True)

ax.plot(u_grid, lam, label=r"$\lambda(u)$")
for idx, Mm in enumerate(M_mark):
    ax.plot(u_mark, G[Mm], marker="o", linestyle="None", label=rf"$\mathcal{{G}}_M(u)$, $M={Mm}$", markersize=2*(idx+1), zorder=10-idx)
ax.plot(u_grid, 1.0 - C1*u_grid, linestyle="--", label=r"$1-(\pi^2/4)u$")
ax.set_xlabel(r"$u$")
ax.set_ylabel(r"$\mathcal{G}_M(u)$")
ax.set_xlim(0, 2.0)
ax.set_ylim(0, 1.02)
ax.legend(loc="upper right", frameon=False, fontsize=11)

outpath = OUTDIR / "fig_lambda_u.png"
fig.savefig(outpath, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", outpath.resolve())
